## 1. Overfitting: menghafal data latihan

**Overfitting** terjadi ketika jaringan terlalu banyak mempelajari detail khusus atau *noise* pada data training, alih-alih hanya mempelajari pola umum yang berguna untuk data baru. Akibatnya, akurasi training dapat sangat tinggi, tetapi performa pada data testing lebih rendah.


## 2. Menyiapkan dataset MNIST

MNIST berisi gambar angka tulisan tangan. Setiap gambar berukuran 28 × 28 piksel, sehingga diratakan menjadi 784 nilai input. Label angka 0–9 diubah menjadi representasi *one-hot* dengan 10 posisi.

Kode persiapan data berikut

In [1]:
import sys, numpy as np
from keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

images, labels = (x_train[0:1000].reshape(1000,28*28) / 255, y_train[0:1000])
one_hot_labels = np.zeros((len(labels),10))

for i,l in enumerate(labels):
    one_hot_labels[i][l] = 1
labels = one_hot_labels

test_images = x_test.reshape(len(x_test),28*28) / 255
test_labels = np.zeros((len(y_test),10))

for i,l in enumerate(y_test):
    test_labels[i][l] = 1

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


### Apa yang dilakukan persiapan data?

- `reshape(1000,28*28)` mengubah setiap gambar menjadi vektor dengan 784 fitur.
- Pembagian dengan `255` menormalkan intensitas piksel ke rentang 0–1.
- `one_hot_labels` menyimpan label dalam bentuk vektor 10 elemen.
- `images` dan `labels` berisi 1.000 contoh training, sedangkan `test_images` dan `test_labels` berisi data testing MNIST.

**Catatan:** Kode ini memakai `keras.datasets.mnist`

## 3. Jaringan tiga layer pada MNIST

Sebelum membahas cara mengurangi overfitting, jaringan bisa mempelajari seluruh 1.000 gambar training dengan sangat baik. Namun, kemampuan menghafal data training tidak menjamin prediksi yang baik pada gambar yang belum pernah dilihat.


In [2]:
np.random.seed(1)

relu = lambda x:(x>=0) * x
relu2deriv = lambda x: x>=0

alpha, iterations, hidden_size, pixels_per_image, num_labels = (0.005, 350, 40, 784, 10)
weights_0_1 = 0.2*np.random.random((pixels_per_image,hidden_size)) - 0.1
weights_1_2 = 0.2*np.random.random((hidden_size,num_labels)) - 0.1

for j in range(iterations):
    error, correct_cnt = (0.0, 0)

    for i in range(len(images)):
        layer_0 = images[i:i+1]
        layer_1 = relu(np.dot(layer_0,weights_0_1))
        layer_2 = np.dot(layer_1,weights_1_2)
        error += np.sum((labels[i:i+1] - layer_2) ** 2)
        correct_cnt += int(np.argmax(layer_2) == np.argmax(labels[i:i+1]))
        layer_2_delta = (labels[i:i+1] - layer_2)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    sys.stdout.write("\r" +
                     " I:" + str(j) +
                     " Error:" + str(error/float(len(images)))[0:5] +
                     " Correct:" + str(correct_cnt/float(len(images))))

 I:349 Error:0.108 Correct:1.0

## 4. Early stopping

**Early stopping** adalah menghentikan pelatihan ketika performa pada data yang tidak dipakai untuk memperbarui bobot mulai memburuk. Ide dasarnya ada pada tahap awal, jaringan cenderung mempelajari pola besar dan umum. semakin lama dilatih, jaringan bisa mulai menangkap detail kecil yang hanya cocok dengan data training.

Dalam praktiknya, pantau performa validation/testing sepanjang training. Jika performa pada data tersebut memburuk meskipun training terus membaik, itu dapat menjadi tanda overfitting. Jangan menggunakan data testing berulang kali untuk memilih model final dalam evaluasi yang ketat. data validation lebih tepat untuk memilih kapan berhenti.

## 5. Dropout: mematikan sebagian hidden nodes secara acak

Dropout adalah cara untuk membuat jaringan besar berperilaku seperti banyak jaringan yang lebih kecil. Pada contoh ini, sekitar 50% node hidden layer dimatikan secara acak saat training.

Tiga operasi penting yang ditambahkan adalah:

1. Membuat `dropout_mask` yang berisi angka 0 dan 1.
2. Mengalikan `layer_1` dengan `dropout_mask * 2`, sehingga sebagian node dimatikan dan aktivasi node yang tersisa diskalakan.
3. Mengalikan `layer_1_delta` dengan `dropout_mask`, agar node yang dimatikan tidak ikut memperbarui bobot melalui jalur tersebut.


In [3]:
i = 0
layer_0 = images[i:i+1]
dropout_mask = np.random.randint(2,size=layer_1.shape)

layer_1 *= dropout_mask * 2
layer_2 = np.dot(layer_1, weights_1_2)

error += np.sum((labels[i:i+1] - layer_2) ** 2)

correct_cnt += int(np.argmax(layer_2) == np.argmax(labels[i:i+1]))

layer_2_delta = (labels[i:i+1] - layer_2)
layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)

layer_1_delta *= dropout_mask

weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

**Catatan** cuplikan di atas adalah potongan dari loop training, bukan sel mandiri yang dapat dijalankan dari awal tanpa variabel sebelumnya. Variabel seperti `layer_1`, `error`, `correct_cnt`, dan bobot harus sudah dibuat oleh kode training sebelumnya.


## 6. Batch gradient descent

Pada *stochastic gradient descent* (SGD), bobot diperbarui setelah setiap contoh. Pada **batch gradient descent**, jaringan menghitung pembaruan berdasarkan sekumpulan contoh, kemudian memperbarui bobot setelah batch selesai.

Listing berikut menunjukkan parameter dan loop batch yang digunakan pada bagian *Batch gradient descent*. Ini adalah contoh kode

In [4]:
import numpy as np
np.random.seed(1)

def relu(x):
    return (x >= 0) * x

def relu2deriv(output):
    return output >= 0

batch_size = 100
alpha, iterations = (0.001, 300)
pixels_per_image, num_labels, hidden_size = (784, 10, 100)

weights_0_1 = 0.2*np.random.random((pixels_per_image,hidden_size)) - 0.1
weights_1_2 = 0.2*np.random.random((hidden_size,num_labels)) - 0.1

for j in range(iterations):
    error, correct_cnt = (0.0, 0)
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = ((i * batch_size),((i+1)*batch_size))

        layer_0 = images[batch_start:batch_end]
        layer_1 = relu(np.dot(layer_0,weights_0_1))
        dropout_mask = np.random.randint(2,size=layer_1.shape)
        layer_1 *= dropout_mask * 2
        layer_2 = np.dot(layer_1,weights_1_2)

        error += np.sum((labels[batch_start:batch_end] - layer_2) ** 2)
        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k+1]) ==
                               np.argmax(labels[batch_start+k:batch_start+k+1]))

        layer_2_delta = (labels[batch_start:batch_end]-layer_2) / batch_size
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * relu2deriv(layer_1)
        layer_1_delta *= dropout_mask

        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)